# Ottimizzazione stocastica con SDDP

Questo notebook ha come obiettivo quello di ottimizzare una rete PyPSA stocastica sfruttando il solver SDDP (Stochastic Dual Dynamic Programming) di SMS++, usando una pipeline di trasformazione _pypsa2smspp_.

Il workflow sarà il seguente:
1. Caricheremo una rete PyPSA stocastica esistente (3 scenari, 4 unità idroelettriche e 100 snapshots);
2. Partizioneremo l'orizzonte in 4 _SDDP stages_ (25 snapshots per stage);
3. Creeremo il blocco SMS++ _SDDPBlock_ usando _pypsa2smspp_;
4. Ottimizzeremo il risultato della pipeline usando _sddp_solver_;
5. Ispezioneremo i seguenti risultati: _objective value_, convergenza e _BellmanValues_. L'_objective value_ sarà estratto dal _solver log_, mentre gli ultimi due saranno dei file a sé stanti prodotti dal solver e localizzabili nella _workdir_ della pipeline.

Non è presente il mapping della soluzione in una soluzione PyPSA, perché quest'ultima attualmente non supporta il caso di reti stocastiche oltre il secondo stadio.

**N.B.** La pipeline che useremo richiede che la rete utilizzata abbia almeno un _StorageUnit_ di tipo _hydro_.

Per eseguire questo workflow è necessario possedere _smspp-project >= 0.7.0_ in ambiente Linux/Ubuntu o macOS e _pysmspp_.

In [ ]:
from pathlib import Path
import shutil
import time

import numpy as np
import pandas as pd
import pypsa
import netCDF4
import pypsa2smspp

from pypsa2smspp.transformation import Transformation

## Settings

Configurazione necessaria affinché avvenga l'esecuzione di SDDP:
- **`SNAPSHOTS_PER_STAGE`**: quantità di PyPSA snapshots appartengono ad ogni stage SDDP;
- **`INITIAL_STATE_FRACTION`**: frazione di _MaxVolumetric_ usato come V0;
- **`STOCHASTIC_PARAMETERS`**: quali dati sono _scenario-dependent_;
- **`SOLVER_PATH`**: path al _sddp_solver_;
- **`CONFIG_FILES`**: i 10 file di configurazione richiesti da SMS++.

In [ ]:
CASE_NAME = "sddp_stochastic_example"

# Network
NETWORK_PATH = Path("notebook_workdir/network_stochastic_network_small_fewsectors__snap100__scen3.nc")
WORKDIR = Path("notebook_workdir")

SNAPSHOTS_PER_STAGE = 25          # 100 / 25 = 4 stages
INITIAL_STATE_FRACTION = 0.5      # V0 = 0.5 * MaxVolumetric
STOCHASTIC_PARAMETERS = ["demand", "hydro_inflow"]

# Solver
SOLVER_PATH = None  

CONFIG_FILES = [
    "SDDPSCfg.txt", "SDDPBCfg.txt", "PFBCfg.txt", "SUBCfg.txt",
    "BUBCfg.txt", "TUBCfg-OVC.txt", "DCNBCfg.txt", "ACBCfg.txt",
    "BCfg.txt", "UCBSCfg.txt",
]
CONFIG_SOURCE = Path("notebook_workdir/configs")  # da dove prendere i file se necessario

## Caricamento della rete stocastica

La rete ha:
- 100 snapshots;
- 3 scenari equiprobabili (_scenario_001_, _scenario_002_, _scenario_003_);
- 12 _StorageUnits_, di cui 4 sono hydro/PHS per scenario;
- 36 generatori, 93 links, 45 buses.

Come prima cosa, è necessario verificare che la rete sia stocastica e che abbia _hydro storage_.

In [ ]:
n = pypsa.Network(str(NETWORK_PATH))

print(f"Snapshots:    {len(n.snapshots)}")
print(f"has_scenarios: {getattr(n, 'has_scenarios', False)}")
print(f"Scenarios:    {list(getattr(n, 'scenarios', []))}")
print(f"Buses:        {len(n.buses)}")
print(f"Generators:   {len(n.generators)}")
print(f"StorageUnits: {len(n.storage_units)}")
print(f"Stores:       {len(n.stores)}")
print(f"Links:        {len(n.links)}")

In [ ]:
# Effettuiamo il controllo sulla presenza di un carrier di tipo hydro
hydro_carriers = {"hydro", "phs"}
carriers = n.storage_units["carrier"].astype(str).str.strip().str.lower()
hydro_mask = carriers.isin(hydro_carriers)

n_hydro = int(hydro_mask.sum())
print(f"Hydro/PHS storage units: {n_hydro}")

if n_hydro == 0:
    raise ValueError(
        "SDDP richiede almeno uno StorageUnit con carrier 'hydro' o 'PHS'."
    )

# Controlliamo che il numero di snapshots sia divisibile per SNAPSHOTS_PER_STAGE
n_snapshots = len(n.snapshots)
if n_snapshots % SNAPSHOTS_PER_STAGE != 0:
    raise ValueError(
        f"Snapshots ({n_snapshots}) deve essere divisibile per "
        f"SNAPSHOTS_PER_STAGE ({SNAPSHOTS_PER_STAGE})."
    )

n_stages = n_snapshots // SNAPSHOTS_PER_STAGE
print(f"SDDP stages: {n_stages} (ognuno con {SNAPSHOTS_PER_STAGE} snapshots)")

## Prepariamo la workdir

SDDP richiede 10 file di configurazione di SMS++ nella directory di lavoro. Controlliamo che siano nella workdir e, se così non fosse, andiamo a copiarli da **CONFIG_SOURCE**. 
Verifichiamo, inoltre, che il solver sia raggiungibile.

In [ ]:
WORKDIR.mkdir(parents=True, exist_ok=True)

# Copiamo i file di configurazione, se mancanti
for fname in CONFIG_FILES:
    dest = WORKDIR / fname
    if not dest.exists():
        src = CONFIG_SOURCE / fname
        if src.exists():
            shutil.copy(src, dest)
        else:
            raise FileNotFoundError(
                f"Il file di configurazione {fname} è assente da {WORKDIR} "
                f"e non è stato trovato in {CONFIG_SOURCE}."
            )

print(f"[ok] {len(CONFIG_FILES)} file di configurazione in {WORKDIR}")

# Verifichiamo che il solver sia raggiungibile
solver = SOLVER_PATH or shutil.which("smspp_sddp_solver")
if solver is None:
    solver = shutil.which("sddp_solver")
if solver is None:
    raise FileNotFoundError(
        "SDDP solver non trovato. Impostare SOLVER_PATH or assicurarsi che `smspp_sddp_solver` "
        "sia nel PATH (attivare l'ambiente conda come prima cosa)."
    )
print(f"[ok] Solver: {solver}")

## Costruiamo l'SDDPBlock

Creiamo una nuova istanza di _Transformation_, che contiene le seguenti configurazioni per SDDP:
- **`stochastic_type = "sddp"`**;
- **`snapshots_per_stage`** partiziona i 100 snapshots in 4 stadi;
- **`top_path.K = 1`** (un _PolyhedralFunctionBlock_ per _HydroSystemUnitBlock_);
- **`initial_state_fraction`** imposta V0 come una frazione di _MaxVolumetric_.

La pipeline genera un `Block_i.nc4` per stadio e un `SDDPBlock_*.nc4` con 4 _StochasticBlocks_, ognuno dei quali punti al suo _UCBlock_ esterno.

In [ ]:
stochastic_parameters = {
    "stochastic_type": "sddp",
    "parameters": STOCHASTIC_PARAMETERS,
    "snapshots_per_stage": SNAPSHOTS_PER_STAGE,
    "top_path": {"K": 1},
    "initial_state_fraction": INITIAL_STATE_FRACTION,
}

# Creiamo una copia cosicché non sia modificata la rete originale
n_smspp = n.copy()

pysmspp_options = {"solver_path": solver} if solver else None

t = Transformation(
    stochastic_parameters=stochastic_parameters,
    workdir=str(WORKDIR),
    name=CASE_NAME,
    capacity_expansion_ucblock=True,
    enable_thermal_units=False,
    pysmspp_options=pysmspp_options,
)

t0 = time.time()
t.create_model(n_smspp, verbose=True)
elapsed_model = time.time() - t0
print(f"\nModello costruito in {elapsed_model:.1f}s")

# Ispezioniamo l'SDDPBlock
print(f"\nState size: {t.sddp_data['state_info']['state_size']}")
print(f"Time horizon (stages): {t.sddp_data['dimensions']['TimeHorizon']}")
print(f"Number of scenarios:   {t.sddp_data['dimensions']['NumberScenarios']}")
print(f"Scenario size:         {t.sddp_data['dimensions']['ScenarioSize']}")

## Ispezioniamo i blocchi generati

Controlliamo la struttura degli hydro dentro i `Block_0.nc4`: ogni _HydroUnitBlock_ dovrebbe avere `NumberReservoirs = 1` (un _reservoir_ per storage unit) con `InitialVolumetric = 0.5 * MaxVolumetric` e `Inflows` dai dati dello scenario.

In [ ]:
with netCDF4.Dataset(WORKDIR / "Block_0.nc4", "r") as ds:
    root = ds.groups[next(iter(ds.groups))]
    for g in root.groups.values():
        if g.getncattr("type") != "HydroSystemUnitBlock":
            continue
        for c in g.groups.values():
            if c.getncattr("type") != "HydroUnitBlock":
                continue
            name = c.getncattr("name")
            inflow = float(c.variables["Inflows"][:].sum())
            v0 = float(c.variables["InitialVolumetric"][:].ravel()[0])
            vmax = float(c.variables["MaxVolumetric"][:].ravel()[0])
            print(
                f"  {name:20s} inflow={inflow:10.2f}  "
                f"V0={v0:12.2f}  Vmax={vmax:12.2f}"
            )

## Ottimizziamo con sddp_solver

Il solver esegue le iterazioni SDDP (_forward/backward passes_). Il log, per ogni iterazione, riporta:
- il **lower bound** (o _backward value_);
- l'**upper bound** (o _forward value_);
- l'**accuratezza**.

Il limite di iterazione è impostato in `SDDPSCfg.txt`.

La convergenza è raggiunta quando l'accuratezza è sotto la soglia target oppure quando viene toccato il limite delle iterazioni.

In [ ]:
t0 = time.time()
t.optimize(verbose=True)
elapsed_opt = time.time() - t0

print(f"\nOptimization completed in {elapsed_opt:.1f}s")
print(f"Objective: {t.result.objective_value:.6e}")

## Analisi della convergenza

Il solver scrive la storia delle iterazioni nel log.
Andiamo alla ricerca di `Backward`/ `Forward` / `accuracy` per visualizzare la convergenza.

In [ ]:
import re

log_path = WORKDIR / f"log_{CASE_NAME}.txt"
if not log_path.exists():
    candidates = list(WORKDIR.glob("log_*.txt"))
    if not candidates:
        raise FileNotFoundError(
            f"Nessun log trovato in {WORKDIR}. Il solver ha girato?"
        )
    log_path = candidates[0]
    print(f"[warning] log atteso non trovato, uso {log_path.name}")

with open(log_path, "r") as f:
    log_text = f.read()

with open(log_path, "r") as f:
    log_text = f.read()

pattern = re.compile(
    r"ACCURACY\s+([\d.eE+-]+)\s+Backward\s+([\d.eE+-]+)\s+"
    r"Forward\s+([\d.eE+-]+)\s+p_iter\s+(\d+)"
)
iterations = []
for m in pattern.finditer(log_text):
    iterations.append({
        "iter": int(m.group(4)),
        "accuracy": float(m.group(1)),
        "backward": float(m.group(2)),
        "forward": float(m.group(3)),
    })

df_conv = pd.DataFrame(iterations)
df_conv["gap"] = df_conv["forward"] - df_conv["backward"]
df_conv

## BellmanValues

Il solver scrive le funzioni del costo futuro per ogni stadio (`BellmanValuesAllOUT.nc4`).

Possiamo ispezionare il numero dei tagli per stadio per verificare che l'algoritmo abbia costruito un approssimazione non banale.

In [ ]:
bv_path = WORKDIR / "BellmanValuesAllOUT.nc4"
if bv_path.exists():
    with netCDF4.Dataset(bv_path, "r") as ds:
        print(f"Bellman values file: {bv_path.name}")
        for stage_name, stage_group in ds.groups.items():
            print(f"\n{stage_name}:")
            for var_name, var in stage_group.variables.items():
                print(f"  {var_name}: shape={var.shape}")
else:
    print(f"Bellman values file not found: {bv_path}")

# Summary

Presentiamo adesso un riassunto dell'esecuzione.

In [ ]:
n_iter = len(df_conv)
final_gap = df_conv["accuracy"].iloc[-1] if not df_conv.empty else None

summary = pd.DataFrame([{
    "case": CASE_NAME,
    "objective": t.result.objective_value,
    "stages": len(n.snapshots) // SNAPSHOTS_PER_STAGE,
    "snapshots_per_stage": SNAPSHOTS_PER_STAGE,
    "scenarios": t.sddp_data["dimensions"]["NumberScenarios"],
    "state_size": t.sddp_data["state_info"]["state_size"],
    "iterations": n_iter,
    "final_accuracy": final_gap,
    "elapsed_s": elapsed_model + elapsed_opt,
}])
summary

## Interpretazione dei risultati

Il solver si è fermato al decimo passo (`Terminated due to iter limit`):
il gap tra backward e forward è 0.85% dell'objective, non ancora sotto
la tolleranza target. Questo è dovuto al limite di iterazioni impostato
in `SDDPSCfg.txt` (10), non a un errore. Per ottenere una convergenza più
stretta basterebbe aumentare il limite o allentare la tolleranza.

Il numero di tagli per stage (10 in `BellmanValuesAllOUT.nc4`) corrisponde
al numero di iterazioni: SDDP ha costruito 10 tagli per ognuno dei 3 stadi
non terminali. Lo stadio 3 (ultimo) ha solo `PolyFunction_lb` (il bound
terminale), come atteso: è la condizione finale del problema.